# 3.2 Logistic Regression

**Class 4.** Table 1 showed **whether** the groups differ, one variable at a time. Regression lets us look at **many variables together** and ask: which factors are associated with **having diabetes**, after accounting for the others?

### By the end of this notebook you'll be able to

1. Explain **odds** and **odds ratios (OR)**, and read a 95% confidence interval
2. Fit **univariable** and **multivariable** logistic regression models with `glm()`
3. Explain **confounding**, and see it by adding variables to a model step by step
4. Write a result in plain language

| | |
|---|---|
| ⏱️ **Time** | 45–60 minutes |
| 📂 **Uses** | `final_analysis.csv` |
| 💾 **Saves** | nothing |

**Before you start:**

- Run **2.0 clean_and_join**
- It helps to have done **3.1 table_one**

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Key concepts

### Probability and odds

- **Probability:** the chance something happens. If 20 out of 100 people have diabetes, the probability is 20/100 = **0.20** (20%).
- **Odds:** the chance it happens divided by the chance it doesn't: 20/80 = **0.25** ("1 to 4").

### Odds ratio (OR)

An **odds ratio** compares the odds in two groups: odds in one group ÷ odds in the other.

| OR | Means |
|---|---|
| **> 1** | higher odds of diabetes (e.g. OR 1.5 = 50% higher odds) |
| **= 1** | no association |
| **< 1** | lower odds (e.g. OR 0.8 = 20% lower odds) |

For a **continuous** predictor like BMI, the OR is the change in odds for each **1-unit** increase (one more kg/m²).

### 95% confidence interval (CI)

The CI is the range of values that are compatible with our data. If the CI for an OR **doesn't include 1**, the association is statistically significant at the 0.05 level. A **narrow** CI means a precise estimate; a **wide** one means uncertainty.

### Logistic regression

Logistic regression models a **yes/no outcome**, here having diabetes. In R it's `glm(outcome ~ predictors, family = binomial)`: `glm` is a *generalized linear model*, and `family = binomial` tells it the outcome is yes/no. The model works on the log-odds scale; we convert its results back to **odds ratios** to read them.

### Univariable vs multivariable, and confounding

- A **univariable** (or *crude*) model has **one** predictor: its raw association with the outcome.
- A **multivariable** (or *adjusted*) model has **several**: each OR is the association **holding the others constant**.

A **confounder** is a variable linked to both the predictor and the outcome that can create or hide an association. For example, older people have higher BMI **and** more diabetes, so age can partly explain a BMI–diabetes link. Adjusting for age separates the two.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

In [ ]:
library(broom)   # tidy() turns model output into a data frame

fa <- prepare_analysis_data()

## Code the outcome as 0/1

Logistic regression needs the outcome as **numbers**: **1** when the event happened (has diabetes) and **0** when it didn't. Our flagging variable from notebook 2.0, `diabetes`, is text ("Yes"/"No"), so we make a 0/1 version. We do the same for the two PTSD variables, to use them as predictors. `if_else(condition, value_if_true, value_if_false)` does this in one line.

In [ ]:
fa <- fa %>%
  mutate(
    diabetes_binary       = if_else(diabetes == "Yes", 1, 0),
    ptsd_doctor_binary    = if_else(ptsd_doctor == "Yes", 1, 0),
    ptsd_treatment_binary = if_else(ptsd_treatment == "Yes", 1, 0)
  )

fa %>% count(diabetes, diabetes_binary)

**🔍 Read the output:** every *No* is 0 and every *Yes* is 1. Also check that **both** outcomes have plenty of people: a model needs enough events to learn from.

## Univariable (crude) models

**Model formulas.** R describes models with `outcome ~ predictor1 + predictor2`. Read `~` as "is modelled by".

`univariable()` fits a model with **one** predictor at a time:

- `reformulate(predictor, "diabetes_binary")` builds the formula `diabetes_binary ~ predictor`
- `tidy(exponentiate = TRUE, conf.int = TRUE)` (from **broom**) returns a tidy table of results, converted to **odds ratios** with 95% CIs
- we drop the `(Intercept)` row, which is the baseline odds and not usually of interest

**Categorical predictors** (like `gender`) are compared with a **reference category**, the first level alphabetically unless you change it. The row `genderMale` is the OR for **Male compared with Female**.

In [ ]:
univariable <- function(predictor) {
  glm(reformulate(predictor, "diabetes_binary"), data = fa, family = binomial) %>%
    tidy(exponentiate = TRUE, conf.int = TRUE) %>%
    filter(term != "(Intercept)")
}

map_dfr(c("age", "BMI", "gender", "ptsd_doctor", "ptsd_treatment"), univariable) %>%
  select(term, estimate, conf.low, conf.high, p.value)

**🔍 Read the output:**

| Column | Means |
|---|---|
| `term` | the predictor (and, for categories, which level vs the reference) |
| `estimate` | the **odds ratio** |
| `conf.low`, `conf.high` | the 95% confidence interval |
| `p.value` | the p-value |

**Interpret:** for each predictor, write one sentence. For example: *"Each 1-unit increase in BMI is associated with X% higher odds of diabetes (OR …, 95% CI … to …)."* An OR of 1.06 is 6% higher odds.

These are **unadjusted**: they ignore every other variable.

### ✍️ Your turn

Write a plain-language sentence for the **`ptsd_treatmentYes`** row of your output. Is it statistically significant? How do you know?

<details>
<summary><b>Show answer</b></summary>

A template: *"People receiving treatment for PTSD had **[OR]** times the odds of diabetes compared with those not receiving treatment (95% CI **[low]** to **[high]**)."*

It's statistically significant if the **95% CI doesn't include 1** (equivalently, p < 0.05). Remember this is **unadjusted**: it could be explained by age, BMI, or other factors.

</details>

## Multivariable models, step by step

We add predictors in **blocks** and keep every model, so we can see how the odds ratios change as we adjust for more:

| Model | Adds | Why |
|---|---|---|
| Base | age, gender, race, ethnicity | standard demographic confounders |
| +BMI | BMI | a strong clinical risk factor |
| +BP+LDL | systolic BP, LDL cholesterol | cardiometabolic markers |
| Full | the two PTSD variables | our variables of interest |

`update()` builds a new formula from an old one: `. ~ . + BMI` means "keep the same outcome (`.`) and all the same predictors (`.`), and add BMI".

In [ ]:
form_base  <- diabetes_binary ~ age + gender + race + ethnicity
form_bmi   <- update(form_base, . ~ . + BMI)
form_bp    <- update(form_bmi,  . ~ . + Systolic_BP + LDL_Cholesterol)
form_full  <- update(form_bp,   . ~ . + ptsd_doctor_binary + ptsd_treatment_binary)

models <- list(
  "Base"    = glm(form_base, data = fa, family = binomial),
  "+BMI"    = glm(form_bmi,  data = fa, family = binomial),
  "+BP+LDL" = glm(form_bp,   data = fa, family = binomial),
  "Full"    = glm(form_full, data = fa, family = binomial)
)

## Compare the odds ratios across models

We want one table with the key ORs from all four models. Rather than calling `tidy()` four times and stacking the results by hand, we use a **loop** from the **purrr** package.

**What is purrr?** The tidyverse's toolkit for doing the same thing to every item in a list:

| Function | Does | Returns |
|---|---|---|
| `map()` | runs code on each item | a list |
| `map_dfr()` | runs code on each item and stacks the results | one data frame |
| `imap_dfr()` | the same, and also gives you each item's **name** (`.y`) | one data frame |

Inside `imap_dfr()`, `.x` is each model and `.y` is its name ("Base", "+BMI"…), which we store in a `model` column.

**Why bother?** One short pipe instead of four near-identical blocks: less copying, fewer mistakes, and adding a fifth model only means adding it to the list.

In [ ]:
imap_dfr(models, ~ tidy(.x, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = .y)) %>%
  filter(term %in% c("BMI", "Systolic_BP", "LDL_Cholesterol",
                     "ptsd_doctor_binary", "ptsd_treatment_binary")) %>%
  select(model, term, estimate, conf.low, conf.high, p.value) %>%
  arrange(term, model)

**🔍 Read the output:** each predictor appears once per model it's in, so you can follow its OR as variables are added. (Age, gender, race, and ethnicity are in every model, but we leave them out of this table to keep it short.)

## Discuss

1. **Stability:** does the BMI odds ratio change much as we add variables? A stable OR suggests the other variables don't confound it much.
2. **Change in direction:** do either of the PTSD odds ratios move a lot, or cross 1, once other variables are included? What could explain that? (Think about PTSD severity, access to care, or medications.)
3. **Statistical vs clinical significance:** an OR of 0.996 per mg/dL of LDL can be "significant" but tiny. Is it meaningful?
4. **Association, not causation:** these are observational data. A significant OR shows an **association**, not that one thing causes the other.

### Writing it up

*"After accounting for age, gender, race, ethnicity, BMI, blood pressure, and LDL cholesterol, people receiving treatment for PTSD had **X** times the odds of diabetes compared with those not receiving treatment (95% CI … to …)."*

### ✍️ Your turn

1. The reference category for `race` is whichever comes first alphabetically. Make **White** the reference instead, refit the Full model, and look at the race rows. (Hint: `relevel()`.)
2. Why would it be a **mistake** to add **HbA1c** as a predictor in these models?

<details>
<summary><b>Show answer</b></summary>

1.
```r
fa <- fa %>% mutate(race = relevel(race, ref = "White"))
full_white_ref <- glm(form_full, data = fa, family = binomial)
tidy(full_white_ref, exponentiate = TRUE, conf.int = TRUE) %>%
  filter(str_detect(term, "race"))
```
Each race OR is now compared with **White**. The model's fit doesn't change; only what each OR is compared against.

2. HbA1c is how diabetes is **diagnosed**, so it would "predict" diabetes almost perfectly. That's circular: it's part of the outcome, not a risk factor. Only include predictors that could plausibly come **before** the outcome.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Message | Means |
|---|---|
| *glm.fit: fitted probabilities numerically 0 or 1 occurred* | A predictor separates the groups (almost) perfectly, e.g. HbA1c. Rethink that predictor. |
| *object 'diabetes_binary' not found* | Run the "Code the outcome as 0/1" cell first |
| A huge OR with a very wide CI | Too few people in some category; combine categories |

## ✅ Key takeaways

- Logistic regression models a **yes/no** outcome; results are **odds ratios**
- OR > 1 = higher odds, OR < 1 = lower odds; check whether the **95% CI includes 1**
- **Multivariable** models adjust for confounders; watch how ORs change as you add variables
- Results show **associations**, not causes

**Going further:** `tidy(models$Full, exponentiate = TRUE)` shows every coefficient; `anova(models$Base, models$Full, test = "LRT")` tests whether the added variables improve the model.

**Next:** **3.3 linear_regression**.